# 基于 OpenBMI (Lee2019) 的视觉脑控拼写器（SSVEP / P300）解码算法研究

本 Notebook 用于在 **Google Colab** 环境下完成 OpenBMI（Lee2019）双范式（SSVEP + P300）数据集下载缓存、信号预处理、模型训练与 **Day 1 → Day 2 严格跨天泛化评估**，所有数据集缓存与实验结果均自动持久化保存至 **Google Drive**。

- **推荐运行时**：`代码执行程序 -> 更改运行时类型 -> T4 GPU`
- **云端项目路径**：`/content/drive/MyDrive/BCI_Portfolio/01_OpenBMI_SSVEP_P300_Speller`
- **结果保存路径**：`results/`（自动输出 CSV、JSON 及`benchmark_report.md`）

## 1. 挂载 Google Drive 并定位项目目录

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/BCI_Portfolio/01_OpenBMI_SSVEP_P300_Speller')
assert PROJECT_DIR.exists(), f'未找到项目目录: {PROJECT_DIR}，请确认已上传至 Google Drive。'

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('当前工作目录:', os.getcwd())
print('目录内容:', sorted(os.listdir(PROJECT_DIR)))

## 2. 检查 GPU 环境并安装项目依赖

In [ ]:
!pip install -q -r requirements.txt

import torch
import mne
import moabb
import pyriemann

print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
print(f'MNE: {mne.__version__} | MOABB: {moabb.__version__} | pyRiemann: {pyriemann.__version__}')

## 3. 配置实验规模并下载/缓存 OpenBMI (Lee2019) 数据集到 Google Drive

- `RUN_MODE = 'quick'`：下载并评估前 5 名被试（适合快速验证全链路，约 3–5 分钟）。
- `RUN_MODE = 'full'`：下载并评估全部 54 名被试（完整基准，建议开启 Colab GPU 后台运行）。
- 也可以通过 `CUSTOM_SUBJECTS` 手动指定被试子集（如 `list(range(1, 21))`）。
- 数据集将自动缓存至 `data/mne_data/`，断点重连后无需重复下载。

In [ ]:
from src.utils import load_config, set_seed, setup_logger
from src.data_pipeline import configure_mne_data_dir

RUN_MODE = 'quick'       # 可选: 'quick' (5名被试) 或 'full' (54名被试)
CUSTOM_SUBJECTS = None   # 例如设为 list(range(1, 16)) 覆盖 RUN_MODE

cfg = load_config(PROJECT_DIR / 'configs' / 'config.yaml')
cfg['project']['data_dir'] = str(PROJECT_DIR / cfg['project']['data_dir'])
cfg['project']['results_dir'] = str(PROJECT_DIR / cfg['project']['results_dir'])

set_seed(int(cfg['project']['seed']))
data_cache_dir = configure_mne_data_dir(cfg['project']['data_dir'])

if CUSTOM_SUBJECTS is not None:
    subjects = list(CUSTOM_SUBJECTS)
else:
    subjects = cfg['dataset']['quick_subjects'] if RUN_MODE == 'quick' else cfg['dataset']['full_subjects']

print(f'数据缓存路径: {data_cache_dir}')
print(f'本次评估被试 ({len(subjects)} 名): {subjects}')

In [ ]:
# 下载并校验数据集到云端 data/mne_data 目录
subj_args = ' '.join(str(s) for s in subjects)
!python scripts/download_dataset.py --subjects {subj_args} --paradigm both

## 4. 运行双范式（SSVEP + P300）Day 1 → Day 2 跨天泛化实验

每完成一名被试会自动写入一次 CSV 断点文件到 Google Drive 的 `results/` 目录，防止长时间运行中断导致进度丢失。

In [ ]:
from src.evaluation import run_full_benchmark

logger = setup_logger('openbmi_bci', log_file=PROJECT_DIR / 'results' / 'run.log')
summary = run_full_benchmark(
    subjects=subjects,
    cfg=cfg,
    run_ssvep=True,
    run_p300=True,
)

## 5. 查看云端保存的完整实验结果与基准评测报告

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

results_dir = PROJECT_DIR / 'results'
print('已保存到云端 results/ 的文件:', sorted(os.listdir(results_dir)))

summary_md = results_dir / 'benchmark_report.md'
if summary_md.exists():
    display(Markdown(summary_md.read_text(encoding='utf-8')))

if (results_dir / 'ssvep_harmonic_snr.csv').exists():
    print('\n--- SSVEP 基频与谐波 SNR 均值统计 (dB) ---')
    df_h = pd.read_csv(results_dir / 'ssvep_harmonic_snr.csv')
    display(df_h.groupby('target_freq_hz')[['h1_snr_db', 'h2_snr_db', 'h3_snr_db']].mean().round(2))

if (results_dir / 'p300_erp_stats.csv').exists():
    print('\n--- P300 250-450ms 波幅与潜伏期均值统计 ---')
    df_e = pd.read_csv(results_dir / 'p300_erp_stats.csv')
    display(df_e.groupby('channel')[['target_mean_uv', 'nontarget_mean_uv', 'p300_peak_diff_uv', 'p300_peak_latency_ms']].mean().round(2))